# Lab 2 · Factorized FNO (FFNO) for infrasound boundary-layer propagation — **30 minutes**

**LETMA 2026 Summer School · Lab 2 of 3** — Lab 1: FNO for Burgers · Lab 3: Graph Neural Networks.

**One scientific question.** *A few tens of ground receivers record infrasound from an explosion.
The signals they record were shaped by the effective sound-speed profile of the atmospheric
boundary layer — can a network learn to predict the recorded waveforms directly from that profile,
fast enough to be used inside inversions, and how do we keep such an operator affordable?*

| Time | Section | What you will see |
|---|---|---|
| 00–02 | §0 Setup | device, imports |
| 02–08 | §1 The physics, with every hypothesis | **wavefront animation + ray tracing + hodograph**: favourable vs unfavourable atmospheres |
| 08–11 | §2 The learning problem | profiles → waveforms, splits |
| 11–17 | §3 The FFNO + **Exercise F2 (4 min)** + training | **animation: waveforms emerging from a profile** |
| 17–19 | §3.5 What the latent space learned | the atmosphere code in 2D |
| 19–25 | §4 The loss, its landscape, **the optimizer race** | who reaches the final loss, who stalls |
| 25–30 | §5 Evaluate on unseen atmospheres · §6 takeaways | held-out waveforms, transfer checklist |

**🥾 Optional deep dives** (skip them live; their outputs are all in the executed solution):
§3.5 latent channel fields + code-space interpolation, §4.3 the ML pathology demos
(seeds / imbalance / overfitting / distribution shift).
**How to finish in 30 minutes — read this before starting.** The critical path is the table
above; the ⏱ checkpoints at each section header tell you where you should be. Three safety
mechanisms keep everyone on track, whatever your ML background:

1. **Exercises cannot block you.** Each one has an automatic checker; if your attempt does not
   pass, the reference implementation takes over and the lab continues — a failed exercise
   costs only the minutes you spent on it, never the lab.
2. **🥾 Optional deep dives are skippable** without losing the thread: all their outputs are
   in the executed solution notebook, and the instructor shows them live if time allows.
3. **🧭 boxes are the novice path.** If you are new to ML: read the 🧭 boxes, watch the
   animations, do the exercise, and keep moving. The 📐 boxes are reference material to
   revisit afterwards — nobody is expected to absorb every formula live.

All the training on the critical path together takes seconds on the cluster GPU and under
~2 min on a laptop CPU: the clock is spent on reading and watching, not on waiting.


**How to use this notebook.** 🧭 boxes assume **no ML background**; 📐 boxes state every assumption
as a numbered hypothesis (H1, H2, …) — including the assumptions of the *physical solver* that
generates the data, because "the data" is where most of the hypotheses live in this subject. Lab 1
(§4) introduced the loss-landscape methodology; here we reuse it in a compact form. Runs on Hedy or
a laptop CPU; animations are inline, press ▶ inside each widget.

## §0 · Setup — 00–02 min · ⏱ running by minute 2\n\nRun the two cells; check the device line.

---
**Authors:** E. Noëlé¹²³⁴, J.-P. Gatti⁵, L. Lehmann⁵, C. Millet⁵, D. Clouteau³

**Affiliations:**
¹ AMIAD — ² CEA — ³ LMPS (Laboratoire de Mécanique Paris-Saclay, CNRS/ENS
Paris-Saclay/CentraleSupélec/Université Paris-Saclay) — ⁴ Centre Borelli (ENS Paris-Saclay) —
⁵ CEA/DAM

**LETMA 2026 Summer School** · Machine Learning on Geophysical Data


In [ ]:
# Portable setup: runs unchanged on the Hedy cluster or on a laptop. Run this cell first.
import os, sys
from pathlib import Path
project_root = next((p for p in [Path(os.environ.get('HEDY_PROJECT_ROOT', Path.cwd())),
                                  Path.cwd(), *Path.cwd().parents]
                     if (p / 'fno_tuto').is_dir() and (p / 'gnn_tuto').is_dir()), None)
if project_root is None:
    raise RuntimeError('Open this notebook inside the tutorial project directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# LAB_PLOT_MODE=tk only for interactive SSH/X11 on Hedy (set before launching the kernel).
from IPython import get_ipython
PLOT_MODE = os.environ.get('LAB_PLOT_MODE', 'inline').strip().lower()
get_ipython().run_line_magic('matplotlib', PLOT_MODE)

import math, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

from fno_tuto.plotting import setup_style
setup_style()
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 12,
                     'axes.labelsize': 11, 'legend.fontsize': 10})
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

# Optional Hedy session profiling: active only on the cluster, silently skipped elsewhere.
HEDY_SESSION = None
try:
    from hedy_euporie.profiler import begin_session
    HEDY_SESSION = begin_session('lab_ffno_infrasound_student', backend='torch')
except Exception:
    print('Hedy session profiling not available here (local/laptop run): continuing without it.')

def show_anim(fig, func, frames, interval=200, blit=False):
    """Inline JavaScript animation: works in Jupyter, JupyterLab and Euporie."""
    ani = animation.FuncAnimation(fig, func, frames=frames, interval=interval, blit=blit)
    plt.close(fig)          # suppress the empty static frame
    display(HTML(ani.to_jshtml()))

print(f'Python {sys.version.split()[0]} | torch {torch.__version__} | device {DEVICE}')

In [ ]:
import torch.nn as nn
from fno_tuto.infrasound import Configuration, make_dataset, sample_ceff, ricker_source
from fno_tuto.models.ffno import FFNO2d, FactorizedSpectralConv2d
from fno_tuto.models.fno2d import SpectralConv2d       # for the parameter comparison
from fno_tuto.losses import get_loss
print('imports OK')

## §1 · The physics, with every hypothesis — 02–08 min · ⏱ checkpoint: start §2 at minute 8

Infrasound (frequencies below ~1 Hz) propagates tens of kilometres through the atmosphere. What a
ground receiver records is governed by the **effective sound speed**

$$c_{\mathrm{eff}}(z) = c(z) + u_{\parallel}(z),$$

the static sound speed plus the wind component *along the propagation plane*. In the boundary layer
(the lowest kilometres, where friction and turbulence shape the wind), $u_{\parallel}$ can vary by
tens of m/s: the atmosphere becomes a **waveguide**, bending the sound back towards the ground or
letting it escape — that is what makes surface recordings informative about the boundary layer.

**The forward model we use (a transparent teaching solver).** For each frequency $\omega$, the
narrow-angle **parabolic equation** (PE) marches the complex field $u(r,z,\omega)$ in range:

$$2 i k_0\, \partial_r u = \partial_{zz}u + \big(k(z)^2 - k_0^2\big)u,
\qquad k(z) = \omega / c_{\mathrm{eff}}(z),\quad k_0 = \omega/c_{\mathrm{ref}},$$

by alternating a half diffraction step (FFT in $z$), a full refraction phase (pointwise in $z$),
and a half diffraction step (Strang splitting). The time trace at a receiver is the inverse FFT over
$\omega$ of $u$ at that range, weighted by the source spectrum.

**Every hypothesis, explicitly — physics (H1–H8) first:**

- **H1 (geometry).** 2-D vertical section $(r,z)$; the profile is **range-independent**: $c_{\mathrm{eff}}$
  depends on $z$ only.
- **H2 (effective medium).** Wind enters only through $u_\parallel$; no cross-plane wind, no
  turbulence, no density stratification.
- **H3 (paraxiality).** One-way propagation, narrow angle (roughly $\lesssim 20^\circ$): no
  backscatter, no wide-angle caustics.
- **H4 (harmonic decomposition).** Frequencies propagate independently; band $[0.04, 0.45]$ Hz,
  i.e. the informative band of the source.
- **H5 (boundaries).** Reflecting ground (mirrored grid); a *nonphysical* sponge above 7.5 km
  absorbs outgoing energy so nothing reflects back artificially.
- **H6 (source).** Point-like explosion at 200 m altitude; Ricker wavelet, $f_0=0.12$ Hz.
- **H7 (receivers).** Four ground receivers at 12, 22, 32, 42 km; retarded time axis
  $\tau = t - r/c_{\mathrm{ref}}$.
- **H8 (amplitudes).** The 2-D scheme produces *relative* pressure; we learn relative waveforms,
  not absolute pascal.

… and, on the machine-learning side:

- **H9 (the operator).** $G^\dagger: c_{\mathrm{eff}}(\cdot) \mapsto \{p_j(\tau)\}_{j=1}^{4}$
  with the source and receivers **fixed** — and the four ground receivers are predicted
  **jointly, in one evaluation**: the output is a single (4 receivers $\times$ 384 s) block.
- **H10 (data law).** Profiles are i.i.d. draws from a synthetic ensemble — smooth
  boundary-layer/tropospheric ducting variations **plus the random Gaussian boundary-layer
  perturbations of H2** (this is the variability the operator must learn to attribute); **not**
  observed atmospheres; splits (48 train / 8 validation / 8 test) by sample index.
- **H11 (normalization).** Input standardized with training-set statistics; one global output
  scale (no per-trace normalization — the relative amplitudes carry physics).
- **H12 (metric).** Relative $L^2$ per atmosphere over all four traces.
- **H13 (architecture, §3).** The operator's spectral response is **separable** in the two Fourier
  axes of the output grid; and the waveforms are essentially band-limited in time (H4 again —
  you will check it yourself).

> 🧭 **First time with ML?** This lab has exactly the same skeleton as Lab 1: data pairs → network
> → loss → optimizer → landscape. Only the objects change: the input is a *profile* $c_{\mathrm{eff}}(z)$
> and the output is a set of *signals* $p_j(\tau)$.

In [ ]:
# ---- the parabolic march, written out in full: ONE frequency, snapshots of u(r,z) ----
CFG = Configuration()
z_m = np.linspace(0.0, CFG.z_max_m, CFG.n_alt)
dz = float(z_m[1] - z_m[0])

def march_snapshots(profile_mps, f_hz, every=3):
    """Narrow-angle PE at a single frequency; returns |u| sampled every `every` range steps."""
    c = np.asarray(profile_mps, float)
    z_ext = np.concatenate((z_m, -z_m[-2:0:-1]))          # mirrored ground (H5)
    c_ext = np.concatenate((c, c[-2:0:-1]))
    n_ext = z_ext.size
    kz = 2 * np.pi * np.fft.fftfreq(n_ext, d=dz)
    k0 = 2 * np.pi * f_hz / CFG.c_ref_mps
    k = 2 * np.pi * f_hz / c_ext                        # wavenumber at each altitude
    dif = np.exp(-0.25j * CFG.dr_m * kz**2 / k0)
    refr = np.exp(0.5j * CFG.dr_m * (k**2 - k0**2) / k0)
    sponge = np.exp(-0.014 * np.maximum(np.abs(z_ext) - 7500.0, 0.0)**2 / 2500.0**2)   # H5
    refr = refr * sponge
    src = np.exp(-0.5 * ((np.abs(z_ext) - CFG.source_alt_m) / CFG.source_vertical_width_m)**2)
    u = (src / src.max()).astype(complex)                 # source at 200 m (H6)
    n_steps = int(round(42e3 / CFG.dr_m))                 # out to the last receiver (H7)
    snaps = []
    for step in range(1, n_steps + 1):
        u = np.fft.ifft(np.fft.fft(u) * dif)              # half diffraction
        u *= refr                                         # full refraction phase
        u = np.fft.ifft(np.fft.fft(u) * dif)              # half diffraction
        if step % every == 0:
            snaps.append(np.abs(u[:CFG.n_alt]).copy())    # physical z >= 0 half
    return np.asarray(snaps)
print('PE marcher defined: single frequency, returns |u(r,z)| snapshots every 3 x 500 m')

In [ ]:
# ---- animation 1: favourable vs unfavourable atmospheres, seen with your own eyes ----
z_km = z_m / 1000.0
DATA = make_dataset(n=64, seed=123, config=CFG)   # generated once, reused in section 2
ceff_all = DATA['ceff_mps']
# favourability index: how much faster the surface is than the air above (H2). Positive =
# downward refraction = the boundary layer traps the sound (favourable); negative = the
# surface is the slowest level = upward refraction = unfavourable atmosphere.
fav_index = ceff_all[:, 0] - ceff_all.min(axis=1)
i_fav, i_unf = int(fav_index.argmax()), int(fav_index.argmin())
print(f'favourable   atmosphere #{i_fav}: surface {fav_index[i_fav]:+.1f} m/s faster than aloft')
print(f'unfavourable atmosphere #{i_unf}: surface excess {fav_index[i_unf]:+.1f} m/s '
      f'(the surface IS the slowest level: waves refract upward and escape)')

U_fav = march_snapshots(ceff_all[i_fav], f_hz=CFG.source_f0_hz)
U_unf = march_snapshots(ceff_all[i_unf], f_hz=CFG.source_f0_hz)
n_frames, r_step = U_fav.shape[0], 3 * CFG.dr_m / 1000.0

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
im0 = ax[0].imshow(U_fav[0][:, None].T, extent=[0, r_step, 0, 10], aspect='auto', origin='lower',
                   cmap='magma', vmin=0, vmax=U_fav.max())
im1 = ax[1].imshow(U_unf[0][:, None].T, extent=[0, r_step, 0, 10], aspect='auto', origin='lower',
                   cmap='magma', vmin=0, vmax=U_fav.max())
for a, t in zip(ax, ['favourable atmosphere (trapped)', 'unfavourable atmosphere (escaping)']):
    a.set_title(t); a.set_xlabel('range r (km)')
    a.axhline(2, color='w', ls=':', lw=0.8); a.axhline(1, color='w', ls='--', lw=0.6)
ax[0].set_ylabel('altitude z (km)')
title = fig.suptitle('')

def update(f):
    r_now = (f + 1) * r_step
    for im, U in [(im0, U_fav), (im1, U_unf)]:
        im.set_data(U[:f + 1].T)
        im.set_extent([0, r_now, 0, 10])
    title.set_text(f'PE wavefield |u(r,z)| at f = {CFG.source_f0_hz:.2f} Hz — '
                   f'range marched to r = {r_now:.0f} km')
    return im0, im1, title

show_anim(fig, update, frames=n_frames, interval=180)

> 🧭 **What to notice.** Left: a **favourable atmosphere** — the surface is faster than the air
> above (positive wind near the ground), so refraction bends the wavefront **down**: energy is
> trapped near the ground and the receivers get a long, guided arrival. Right: an
> **unfavourable atmosphere** — the surface is the slowest level, refraction sends energy upward
> (the sponge absorbs it) and receivers see a weaker, more complex signal. *Same source, same
> geometry — the only difference is the input profile you would like to infer.* The white lines
> mark 1 and 2 km: this all happens in the boundary layer.

### 1.1 · The same physics as a ray picture, and as a hodograph

Two classical views make the problem tangible before any machine learning:

- **Ray tracing.** In a stratified medium the ray angle $e$ obeys
  $\frac{de}{ds} = \frac{1}{c}\frac{dc}{dz}\cos e$: rays bend toward the slower medium. The
  ground is a mirror (H5), so each ray that returns is a recorded arrival. Favourable and
  unfavourable atmospheres give opposite ray fans — with the **input profile shown next to each
  fan** (vertical: altitude on the $y$-axis, celerity on the $x$-axis, as in every profile plot
  of this lab).
- **Hodograph.** The recordings themselves, stacked by receiver range: each trace is rescaled
  and placed at its range, so the arrival pattern *is* the travel-time curve. The ray-predicted
  first arrivals (shifted by the source wavelet of H6) land where the traces peak — the reduced
  time $	au = t - r/c_{\mathrm{ref}}$ removes the bulk travel, so what remains is the extra
  path travelled inside the duct.

In [ ]:
# ---- illustration by ray tracing: favourable vs unfavourable atmospheres ----
def trace_rays(c, angles_deg, r_max=45e3, ds=20.0, z_src=CFG.source_alt_m, z_top=10500.0):
    """Rays in a stratified medium: de/ds = (dc/dz)/c * cos(e); the ground is a mirror
    (H5): every ground hit is an arrival (range, reduced time)."""
    dcdz = np.gradient(c, z_m)
    rays, arrivals = [], []
    for e0 in angles_deg:
        e = np.radians(e0); x, z, t = 0.0, float(z_src), 0.0
        xs, zs = [x], [z]
        for _ in range(int(r_max / ds) + 800):
            cc = float(np.interp(z, z_m, c)); g = float(np.interp(z, z_m, dcdz))
            e += (g / cc) * np.cos(e) * ds
            x += np.cos(e) * ds; z += np.sin(e) * ds; t += ds / cc
            if z <= 0.0:                       # ground hit: reflect (Neumann boundary)
                z = -z; e = -e
                arrivals.append((x / 1000.0, t - x / CFG.c_ref_mps))
            if z >= z_top or x >= r_max:
                break
            xs.append(x); zs.append(z)
        rays.append((np.array(xs) / 1000.0, np.array(zs) / 1000.0))
    return rays, np.array(arrivals) if arrivals else np.zeros((0, 2))

angles = np.arange(2.0, 20.1, 2.0)
rays_fav, arr_fav = trace_rays(ceff_all[i_fav].astype(float), angles)
rays_unf, arr_unf = trace_rays(ceff_all[i_unf].astype(float), angles)
print(f'favourable atmosphere:   {len(arr_fav)} ground arrivals within 45 km (the layer traps)')
print(f'unfavourable atmosphere: {len(arr_unf)} ground arrivals (every ray escapes upward)')

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.5), sharey=True)
for a, (rays, i, title) in zip(ax, [(rays_fav, i_fav, 'favourable atmosphere (trapped)'),
                                    (rays_unf, i_unf, 'unfavourable atmosphere (escaping)')]):
    for xr, zr in rays:
        a.plot(xr, zr, lw=0.9, color='C0', alpha=0.5)
    a.plot([0, 45], [0, 0], color='0.25', lw=2)                    # reflecting ground
    a.plot(CFG.station_ranges_km, [0.35] * 4, 'v', color='C3', ms=7)
    for r in CFG.station_ranges_km:
        a.annotate(f'{int(r)}', (r, 1.4), fontsize=8, color='C3', ha='center')
    a.axhline(1.0, color='0.6', ls='--', lw=0.8)
    inset = a.inset_axes([0.80, 0.56, 0.18, 0.40])
    inset.plot(ceff_all[i], z_km, color='C1', lw=1.6)               # input profile (vertical)
    inset.set_ylim(0, 10); inset.set_xticks([]); inset.set_yticks([])
    inset.set_xlabel('c_eff', fontsize=7)
    a.set_xlim(0, 45); a.set_ylim(0, 10.5)
    a.set_xlabel('range r (km)'); a.set_title(title, fontsize=11)
ax[0].set_ylabel('altitude z (km)')
fig.suptitle('ray tracing through c_eff(z): same source, same stations — only the input '
             'profile (insets) differs', y=1.02)
plt.show()

In [ ]:
# ---- the hodograph: arrival time vs receiver range, from the data AND from the rays ----
tau_np = DATA['tau_s']
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.0), sharey=True)
for a, (traces, arr, title) in zip(ax, [(DATA['p'][i_fav], arr_fav,
                                         'favourable: strong, early, simple returns'),
                                        (DATA['p'][i_unf], arr_unf,
                                         'unfavourable: weak, complex returns')]):
    for j, r in enumerate(CFG.station_ranges_km):
        tr = traces[j] / (np.abs(traces[j]).max() + 1e-9)
        a.plot(tau_np, r + 3.0 * tr, color='C0', lw=1.3)
    if len(arr):
        ok = arr[:, 0] <= 45
        a.plot(arr[ok, 0], CFG.source_center_tau_s + arr[ok, 1], 'o', ms=4, color='C3',
               label='ray-predicted arrivals (+ wavelet)')
        a.legend(fontsize=8, loc='upper left')
    a.set_xlabel('retarded time tau (s)'); a.set_title(title, fontsize=11)
ax[0].set_ylabel('receiver range (km)'); ax[0].set_ylim(4, 50)
fig.suptitle('hodograph (time x receiver): each trace is rescaled and stacked at its range; '
             'the arrival trend matches the rays', y=1.02)
plt.show()
print('the traces peak where the ray prediction lands: the few seconds of reduced time are')
print('exactly the extra path travelled inside the layer. Rays are the high-frequency limit')
print('of the PE (H3): the match is qualitative - and that is the point of the illustration.')

>
> 📐 **Rigor checkpoint.** Everything above follows from H1–H8. If a colleague's data violates H1
> (range-dependent wind), H3 (steep angles, e.g. mountains) or H8 (absolute calibration), the
> operator we train below is answering a *different question* than their experiment.

## §2 · The learning problem — 08–11 min · ⏱ checkpoint: start §3 at minute 11

We generate 64 synthetic atmospheres (H10), solve the PE for each, and learn **H9** directly:
profile in, four waveforms out. Note the cost asymmetry that motivates the whole subject: the PE
march takes seconds per atmosphere here (and hours in operational 3-D settings), while the trained
operator will take milliseconds.

In [ ]:
# ---- dataset (H10): the 64 atmospheres already generated in section 1 ----
ceff = torch.tensor(DATA['ceff_mps']).float()          # (64, 64)  profiles c_eff(z)
P = torch.tensor(DATA['p']).float()                    # (64, 4, 384) traces p_j(tau)
tau_s = torch.tensor(DATA['tau_s']).float()
ranges_km = DATA['ranges_km']
print('profiles:', tuple(ceff.shape), '| traces:', tuple(P.shape),
      '| receivers at', [float(r) for r in ranges_km], 'km')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for row in ceff.numpy():
    ax[0].plot(row, z_km, color='C0', alpha=0.15)
ax[0].plot(ceff[i_fav].numpy(), z_km, color='C3', lw=2, label='favourable example')
ax[0].plot(ceff[i_unf].numpy(), z_km, color='C2', lw=2, label='unfavourable example')
ax[0].axhspan(0, 2, color='orange', alpha=0.10, label='boundary layer')
ax[0].set_ylim(0, 10)
ax[0].set_ylabel('altitude z (km)'); ax[0].set_xlabel('c_eff (m/s)')
ax[0].legend(fontsize=8, loc='lower right')
ax[0].set_title('the input ensemble (H10), vertical view')
for j in range(4):
    ax[1].plot(tau_s, P[i_fav, j] / P.abs().max(), color='C3', alpha=0.5 + 0.15 * j,
               label='favourable' if j == 0 else None)
    ax[1].plot(tau_s, P[i_unf, j] / P.abs().max(), color='C2', alpha=0.5 + 0.15 * j,
               label='unfavourable' if j == 0 else None)
ax[1].set_xlabel('retarded time tau (s)'); ax[1].set_ylabel('rel. pressure'); ax[1].legend(fontsize=9)
ax[1].set_title('recorded traces at 12-42 km: phase and duration encode the profile')
plt.tight_layout(); plt.show()

In [ ]:
# ---- the statistical distributions of the INPUTS and the OUTPUTS (know your data!) ----
fig, ax = plt.subplots(1, 3, figsize=(12.8, 3.6))
m_np = ceff.mean(0).numpy(); s_np = ceff.std(0).numpy()
ax[0].fill_betweenx(z_km, m_np - s_np, m_np + s_np, color='C0', alpha=0.25, label='mean +/- std')
ax[0].plot(m_np, z_km, color='C0', lw=1.8)
ax[0].plot(ceff[i_fav].numpy(), z_km, color='C3', lw=1.1, label='favourable')
ax[0].plot(ceff[i_unf].numpy(), z_km, color='C2', lw=1.1, label='unfavourable')
ax[0].axhspan(0, 2, color='orange', alpha=0.10)
ax[0].set_ylim(0, 10); ax[0].set_ylabel('altitude z (km)'); ax[0].set_xlabel('c_eff (m/s)')
ax[0].set_title('INPUT: where the variability lives')
ax[0].legend(fontsize=7, loc='lower right')
ax[1].hist(ceff.numpy().ravel(), bins=48, color='C0', alpha=0.85)
ax[1].axvline(float(ceff.mean()), color='k', ls='--', lw=1)
ax[1].set_xlabel('c_eff value (m/s)'); ax[1].set_ylabel('count')
ax[1].set_title(f'INPUT values: mean {float(ceff.mean()):.1f}, std {float(ceff.std()):.1f} m/s')
ax[2].hist(P.numpy().ravel(), bins=64, color='C3', alpha=0.85)
ax[2].set_yscale('log'); ax[2].set_xlabel('relative pressure'); ax[2].set_ylabel('count (log)')
ax[2].set_title('OUTPUT values: heavy-tailed (loud arrivals, quiet tails)')
plt.tight_layout(); plt.show()
e_amps = P.flatten(1).norm(dim=1)
print(f'input c_eff: mean {float(ceff.mean()):.1f} +/- {float(ceff.std()):.1f} m/s | '
      f'output trace energies spread {float(e_amps.min()):.1f} to {float(e_amps.max()):.1f} '
      f'(x{float(e_amps.max() / e_amps.min()):.1f}) - the imbalance of section 4, in numbers')

In [ ]:
# ---- tensors and splits (H10, H11) ----
NTR, NVA, NTE = 48, 8, 8
c_mu, c_sd = ceff[:NTR].mean(), ceff[:NTR].std()       # input standardization, train stats only
p_sd = P[:NTR].std()                                   # ONE global output scale (H11)
Xc = ((ceff - c_mu) / c_sd).to(DEVICE)                 # (64, 64)
Yn = (P / p_sd).to(DEVICE)                             # (64, 4, 384) normalized

Xtr, Ytr = Xc[:NTR], Yn[:NTR]
Xva, Yva = Xc[NTR:NTR + NVA], Yn[NTR:NTR + NVA]
Xte, Yte = Xc[NTR + NVA:], Yn[NTR + NVA:]
print(f'train {NTR} | validation {NVA} | test {NTE} atmospheres '
      f'(test untouched until section 5)')

## §3 · The FFNO: factorize the Fourier layers — 11–17 min · ⏱ checkpoint: start §4 at minute 17

Lab 1's FNO mixed channels with one complex matrix **per mode**. On a 2-D grid $(x, y)$ a full
spectral layer needs a matrix **per pair of modes**, $R[k_x, k_y]$: for $C$ channels, $m_x{\times}m_y$
pairs, that is $C^2\, m_x m_y$ complex parameters per layer. The **Factorized** FNO (Tran et al.,
ICLR 2023) replaces one 2-D spectral convolution by the **sum of two 1-D ones**:

$$(K v)^{\wedge}[k_x, k_y] = R_x[k_x]\, \hat v[k_x, k_y] + R_y[k_y]\, \hat v[k_x, k_y],$$

with $C^2 (m_x + m_y)$ parameters — a **tensor rank-1 (sum-separable) constraint** on the filter.
Two further ideas make deep stacks affordable: transformer-style residual blocks
$v \leftarrow v + \mathrm{FF}(\mathrm{GELU}(K(\mathrm{Norm}(v))))$ (the identity path keeps
many layers trainable), and optional weight sharing.

**Our operator (H9).** The *input* is a 1-D profile, the *output* is a 2-D grid (receiver × time):

1. **Encode altitude**: a small 1-D conv pool summarizes $c_{\mathrm{eff}}(z)$ into $k=16$ numbers
   per atmosphere.
2. **Decode on the (receiver, time) grid**: broadcast those $k$ channels onto the $4{\times}384$
   grid, add coordinate channels (the FFNO does this internally), and run $L=4$ factorized blocks:
   the spectral conv along **time** synthesizes waveform structure from the profile features
   (H13 + H4: infrasound is band-limited — Exercise F2 checks it), the one along **receivers**
   couples the four ranges (only 2 modes exist there: 4 points).
3. **Project** to 1 channel: the four normalized traces.

> 📐 **What factorization costs, honestly.** A separable filter cannot couple $k_x$ with $k_y$:
> phase terms of the form $e^{i(\alpha k_x + \beta k_y)}$ with *both* axes varying are
> unreachable in one layer. This is a hypothesis (H13), and in exchange you save a factor
> $\approx m_x m_y/(m_x+m_y)$ of spectral parameters — at $m_x=m_y=32$ that is 16×.

**Exercise F2 · 4 minutes.** Complete `signal_spectrum` (one line: two NumPy calls) and check H4/H13
yourself: where does the energy of a recorded trace live in frequency? ⏱ hard stop at minute 15:
not passing yet? The checker falls back to the NumPy reference — read the hint after class.

In [ ]:
import numpy as np

def signal_spectrum(waveform, dt):
    """Real waveform (Nt,) -> (frequencies in Hz, complex one-sided spectrum)."""
    # TODO: return np.fft.rfftfreq(len(waveform), d=dt), np.fft.rfft(waveform)
    raise NotImplementedError('Complete signal_spectrum: one line, two NumPy calls')

In [ ]:
# ---- check F2 against the NumPy reference, then look at the physics it reveals ----
trace = P[i_fav, 3]
f_ref, spec_ref = np.fft.rfftfreq(len(trace), d=CFG.dt_s), np.fft.rfft(trace)
try:
    f_stu, spec_stu = signal_spectrum(trace, CFG.dt_s)
    ok = np.allclose(f_stu, f_ref) and np.allclose(spec_stu, spec_ref)
except Exception as exc:
    ok = False
    print('your function raised:', type(exc).__name__, exc)
print('F2:', 'your spectrum is correct' if ok else 'not correct yet - using the reference below')

trace2 = P[i_unf, 3]
fig, ax = plt.subplots(1, 3, figsize=(12.8, 3.2))
ax[0].plot(ceff[i_fav].numpy(), z_km, color='C3', label='favourable')
ax[0].plot(ceff[i_unf].numpy(), z_km, color='C2', label='unfavourable')
ax[0].set_ylim(0, 10); ax[0].set_ylabel('altitude z (km)'); ax[0].set_xlabel('c_eff (m/s)')
ax[0].legend(fontsize=8); ax[0].set_title('the two input profiles')
ax[1].plot(f_ref, np.abs(spec_ref), color='C3', label='favourable')
ax[1].plot(f_ref, np.abs(np.fft.rfft(trace2)), color='C2', label='unfavourable')
ax[1].axvspan(CFG.f_min_hz, CFG.f_max_hz, color='0.8', alpha=0.5,
              label='band propagated by the solver (H4)')
ax[1].set_xlabel('frequency (Hz)'); ax[1].set_ylabel('|P(f)|'); ax[1].legend(fontsize=8)
ax[1].set_title('trace spectra at 42 km')
ax[2].plot(tau_s, trace, color='C3', label='favourable')
ax[2].plot(tau_s, trace2, color='C2', label='unfavourable')
ax[2].set_xlabel('retarded time tau (s)'); ax[2].set_ylabel('rel. pressure'); ax[2].legend(fontsize=8)
ax[2].set_title('the same traces (time domain)')
plt.tight_layout(); plt.show()
print('H13 in the time direction: keeping ~64 of 193 time modes covers the whole band — '
      'exactly the truncation the FFNO will make.')

In [ ]:
# ---- the operator of H9, assembled from the packaged FFNO2d ----
class InfrasoundFFNO(nn.Module):
    """c_eff(z) -> (4, 384) normalized traces: altitude encoder + FFNO over (receiver, time)."""
    def __init__(self, k=16, width=16, modes=(2, 64), n_layers=4):
        super().__init__()
        self.k = k
        self.encoder = nn.Sequential(                       # 1-D conv + pool along altitude
            nn.Conv1d(1, 32, 5, padding=2), nn.GELU(),
            nn.AdaptiveAvgPool1d(8), nn.Flatten(), nn.Linear(32 * 8, k), nn.GELU())
        self.ffno = FFNO2d(in_channels=k, out_channels=1, width=width, modes=modes,
                           n_layers=n_layers, grid=True, proj_hidden=64)

    def forward(self, c):                                  # c: (B, 64) standardized profile
        h = self.encoder(c.unsqueeze(1))                   # (B, k) summary of the atmosphere
        v = h.view(-1, self.k, 1, 1).expand(-1, self.k, 4, 384)   # broadcast onto the grid
        return self.ffno(v).squeeze(1)                     # (B, 4, 384) normalized traces

torch.manual_seed(0)
ffno = InfrasoundFFNO()
n_par = sum(p.numel() * (2 if p.is_complex() else 1) for p in ffno.parameters())
# The factorization, in parameters, at our sizes (C = width = 16, mx = 2, my = 64):
C, mx, my = 16, 2, 64
full = C * C * mx * my
fact = C * C * (mx + my)
print(f'FFNO operator: {n_par:,} real parameters')
print(f'spectral parameters per layer at (mx, my)=({mx},{my}): full FNO {full:,} vs '
      f'factorized {fact:,} complex weights -> {full/fact:.0f}x fewer')

In [ ]:
# ---- training loop (same skeleton as Lab 1: AdamW + cosine, instrumented) ----
def fit_ffno(model, *, epochs=120, batch=16, lr=2e-3, wd=1e-5, loss_name='rel_l2',
             snap_every=12, n_snap=3, seed=0):
    torch.manual_seed(seed)
    loss_fn = get_loss(loss_name)
    decay, no_decay = [], []
    for p in model.parameters():
        (no_decay if (p.is_complex() or p.ndim <= 1) else decay).append(p)
    opt = torch.optim.AdamW([{'params': decay, 'weight_decay': wd},
                             {'params': no_decay, 'weight_decay': 0.0}], lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * math.ceil(NTR / batch))
    model = model.to(DEVICE)
    hist = {'train': [], 'val': []}
    snaps = []
    gen = torch.Generator().manual_seed(seed)
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(NTR, generator=gen)
        tot, nb = 0.0, 0
        for i in range(0, NTR, batch):
            idx = perm[i:i + batch]
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(Xtr[idx]), Ytr[idx])
            loss.backward(); opt.step(); sched.step()
            tot += loss.item(); nb += 1
        model.eval()
        with torch.no_grad():
            val = get_loss('rel_l2')(model(Xva), Yva).item()
        hist['train'].append(tot / nb); hist['val'].append(val)
        if snap_every and (ep % snap_every == 0 or ep == 1):
            with torch.no_grad():
                snaps.append((ep, model(Xva[:n_snap]).cpu().numpy()))
    return hist, snaps

theta_init_ffno = None
torch.manual_seed(0)
ffno = InfrasoundFFNO()
hist, snaps = fit_ffno(ffno, epochs=120)
print(f"final validation rel-L2 (normalized traces): {hist['val'][-1]:.4f} "
      f"(recall: 1.0 = predicting zero)")
fig, ax = plt.subplots(figsize=(6.8, 3))
ax.plot(hist['train'], label='training loss')
ax.plot(hist['val'], label='validation rel-L2')
ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('120 epochs, ~45 s on a laptop CPU (seconds on a GPU)')
plt.show()

In [ ]:
# ---- animation 2: waveforms emerging from a profile, epoch by epoch ----
ep0, pr0 = snaps[0]
fig, ax = plt.subplots(1, 6, figsize=(15.5, 3.0))
truth = P[NTR:NTR + 3].numpy()
ax[0].plot(Xva[0].cpu().numpy() * float(c_sd) + float(c_mu), z_km, color='C1', lw=1.8)
ax[0].set_ylim(0, 10); ax[0].set_xlabel('c_eff (m/s)', fontsize=8)
ax[0].set_ylabel('altitude (km)', fontsize=8)
ax[0].set_title('INPUT profile', fontsize=9)
ax[0].axhspan(0, 2, color='orange', alpha=0.10)
lines = []
for j in range(4):
    ax[1 + j].plot(tau_s, truth[0, j], color='C0', lw=1.8, label='PE truth')
    (ln,) = ax[1 + j].plot(tau_s, pr0[0, j] * float(p_sd), color='C3', ls='--', lw=1.5, label='FFNO')
    ax[1 + j].set_title(f'{int(ranges_km[j])} km', fontsize=9)
    ax[1 + j].set_xlabel('tau (s)', fontsize=8); ax[1 + j].grid(alpha=0.3)
    lines.append(ln)
ax[1].set_ylabel('rel. pressure'); ax[1].legend(fontsize=7)
(loss_ln,) = ax[5].plot([s[0] for s in snaps], [hist['val'][s[0] - 1] for s in snaps], color='0.4')
(dot,) = ax[5].plot([], [], 'o', color='C3')
ax[5].set_xlabel('epoch'); ax[5].set_ylabel('val rel-L2'); ax[5].grid(alpha=0.3)
ax[5].set_yscale('log'); ax[5].set_title('training', fontsize=9)
title = fig.suptitle('', y=1.04)

def update(f):
    ep, pr = snaps[f]
    for j in range(4):
        lines[j].set_ydata(pr[0, j] * float(p_sd))
    dot.set_data([ep], [hist['val'][ep - 1]])
    title.set_text(f'epoch {ep}: one atmosphere in, ALL FOUR ground receivers out (predicted jointly)')

show_anim(fig, update, frames=len(snaps), interval=450)

> 🧭 **What to notice.** The network first learns the **arrival time** (phase), then the envelope,
> then the interference detail. Phase is the hard part: a travel-time error of a few percent is an
> $O(1)$ waveform error even if every amplitude is right. This is why waveform losses are
> delicate — and it motivates §4.

### 3.5 · Why we encode at all: the operator as a factorization through a latent space — 17–19 min · ⏱ checkpoint: start §4 at minute 19

The architecture above is usually drawn as boxes; here is what it *is*, as one composed map:

$$\hat p = Q \circ B_L \circ \cdots \circ B_1 \circ (h \otimes 1) \circ E,
\qquad E: c_{\mathrm{eff}} \mapsto h \in \mathbb R^{k}, \qquad k = 16 .$$

- **$E$, the encoder**, compresses the 64-point profile into $k = 16$ numbers — the
  **atmosphere code**. The broadcast $h \otimes 1$ paints that code onto the (receiver, time)
  grid as a constant field: it is **the only channel through which the atmosphere enters the
  decoder**.
- **$B_1 \dots B_L$** act on the **latent field** $v$: each grid point carries $C = 16$ learned
  channels; the blocks mix channels per Fourier mode (spectral convolutions) and per grid point
  (the MLPs), with residual updates.
- **$Q$** reads the latent field back to physical pressure.

**Why must something like $E$ exist?** The input lives on the altitude axis; the output lives on
the (receiver, time) grid. These domains never touch inside the network — no spectral
convolution connects the $z$-axis of the input to the $(r, 	au)$ axes of the output — so some
map has to transport the atmospheric information across. The code $h$ is that bridge: by
construction, *the decoder sees the whole atmosphere through 16 numbers*.

> 📐 **H14 (bottleneck hypothesis).** We assume $G^\dagger$ factors through a $k$-dimensional
> code: two atmospheres with the same $h$ receive identical predicted waveforms. This is a
> falsifiable approximation, not a technicality: $k$ controls an information bottleneck
> (larger $k$ = more capacity but more data needed; smaller $k$ = compression that can delete
> physics). And the latent space is a **hypothesis class, not an interpretable object by
> default**: what the 16 numbers *mean* is decided by training alone. The next three cells
> inspect what was actually learned.

In [ ]:
# ---- what the encoder learned: the atmosphere code, seen in 2D (PCA) ----
ffno.eval()
with torch.no_grad():
    codes = ffno.encoder(Xc.unsqueeze(1)).cpu().numpy()     # (64, 16): one code per atmosphere
codes_c = codes - codes.mean(0)
_, s, vt = np.linalg.svd(codes_c, full_matrices=False)
PROJ = codes_c @ vt[:2].T                                   # first two principal components
explained = (s[:2] ** 2).sum() / (s ** 2).sum()

split_marker = ('o', 's', '^')                               # train / validation / test
split_id = np.zeros(64, int); split_id[48:56] = 1; split_id[56:] = 2
fig, ax = plt.subplots(figsize=(6.6, 5.2))
for sp in range(3):
    idx = np.flatnonzero(split_id == sp)
    sc = ax.scatter(PROJ[idx, 0], PROJ[idx, 1], c=fav_index[idx], cmap='coolwarm',
                    s=28 if sp == 0 else 60, marker=split_marker[sp],
                    edgecolor='k', lw=0.3, label=('train', 'validation', 'test')[sp])
fig.colorbar(sc, ax=ax, label='favourability: c_eff(0) - min c_eff (m/s)')
ax.set_xlabel('principal component 1'); ax.set_ylabel('principal component 2')
ax.legend(fontsize=9, title='split'); ax.grid(alpha=0.3)
ax.set_title(f'the learned atmosphere code (16-D, shown in 2D); '
             f'PC1-2 explain {explained * 100:.0f}% of its variance')
plt.show()
corr = np.corrcoef(PROJ[:, 0], fav_index)[0, 1]
print(f'correlation between principal component 1 and the physical jet strength: {corr:+.2f}')
print('the code is NOT a copy of the profile: it is a learned 16-D representation in which')
print('atmospheres are placed mainly according to how favourable they are for propagation -')
print('physics the network was never told about.')

In [ ]:
# ---- 🥾 OPTIONAL (skip live if short on time): the learned Fourier weights ----
def spectral_responses(model):
    """Mean |R| per Fourier mode for every block: the learned transfer functions."""
    out = []
    for li, blk in enumerate(model.ffno.blocks, 1):
        wx = blk.spectral.weight_x.detach().abs().mean(dim=(0, 1)).cpu().numpy()
        wy = blk.spectral.weight_y.detach().abs().mean(dim=(0, 1)).cpu().numpy()
        out.append((li, wx, wy))
    return out

resp = spectral_responses(ffno)
freqs = np.arange(len(resp[0][2])) / (CFG.n_time * CFG.dt_s)   # mode k <-> f = k / (Nt dt) (H4)
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))
for li, wx, wy in resp:
    ax[0].plot(freqs, np.maximum(wy, 1e-9), lw=1.6, label=f'block {li}')
ax[0].axvspan(CFG.f_min_hz, CFG.f_max_hz, color='0.85', zorder=0,
              label='infrasound band (H4)')
ax[0].set_yscale('log')
ax[0].set_xlabel('frequency f = k / (Nt dt)  (Hz)')
ax[0].set_ylabel('mean |R_y[k]|  (log)')
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[0].set_title('time-axis Fourier weights: the blocks learned a band-pass')
for li, wx, wy in resp:
    ax[1].plot(np.arange(len(wx)), wx, 'o-', ms=4, label=f'block {li}')
ax[1].set_xlabel('receiver-axis mode k (only 2 exist)')
ax[1].set_ylabel('mean |R_x[k]|')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
ax[1].set_title('receiver-axis Fourier weights')
plt.tight_layout(); plt.show()
for li, wx, wy in resp:
    k_peak = int(np.argmax(wy[1:]) + 1)
    print(f'block {li}: peak response at time-mode k = {k_peak} '
          f'(f = {k_peak / (CFG.n_time * CFG.dt_s):.3f} Hz) — inside the infrasound band')
print('the trainable spectra concentrate where the signal is (H4) and roll off beyond:')
print('the spectral convolutions act as learned band-pass filters — this is what the')
print('"latent space of channels" means physically: a bank of complex Fourier weights.')

In [ ]:
# ---- 🥾 OPTIONAL: the code as a coordinate system (interpolation animation) ----
with torch.no_grad():
    h_a = ffno.encoder(Xva[0:1].unsqueeze(1))
    h_b = ffno.encoder(Xva[5:6].unsqueeze(1))

def decode_from_code(h):
    with torch.no_grad():
        v = h.view(-1, ffno.k, 1, 1).expand(-1, ffno.k, 4, 384)
        return ffno.ffno(v).squeeze(1)

j = 3                                           # the 42 km receiver
alphas = np.linspace(0, 1, 21)
tr_a = (decode_from_code(h_a)[0, j] * float(p_sd)).cpu().numpy()
p_a, p_b = PROJ[48], PROJ[53]                   # the two codes in the PCA plane

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))
ax[0].plot(tau_s, P[48, j].numpy(), color='C0', lw=1.8, label='atmosphere A (truth)')
ax[0].plot(tau_s, P[53, j].numpy(), color='C2', lw=1.8, label='atmosphere B (truth)')
(ln,) = ax[0].plot(tau_s, tr_a, color='C3', ls='--', lw=1.8,
                   label='decoded from interpolated code')
ax[0].set_xlabel('tau (s)'); ax[0].set_ylabel('rel. pressure'); ax[0].legend(fontsize=8)
ax[0].set_title('42 km trace decoded from (1-a) h_A + a h_B')
ax[1].scatter(PROJ[:, 0], PROJ[:, 1], s=10, c='0.8')
ax[1].plot([p_a[0], p_b[0]], [p_a[1], p_b[1]], '-', color='C3', lw=1)
ax[1].plot([p_a[0]], [p_a[1]], 'o', color='C0', ms=8)
ax[1].plot([p_b[0]], [p_b[1]], 'o', color='C2', ms=8)
(dot,) = ax[1].plot([], [], 'o', color='C3', ms=9)
ax[1].set_xlabel('PC 1'); ax[1].set_ylabel('PC 2'); ax[1].grid(alpha=0.3)
ax[1].set_title('the walk in the learned code space')
title = fig.suptitle('', y=1.02)

def update(f):
    a = alphas[f]
    h = (1 - a) * h_a + a * h_b
    ln.set_ydata((decode_from_code(h)[0, j] * float(p_sd)).cpu().numpy())
    dot.set_data([(1 - a) * p_a[0] + a * p_b[0]], [(1 - a) * p_a[1] + a * p_b[1]])
    title.set_text(f'interpolation parameter a = {a:.2f}: the code morphs from A (blue) to B (green)')

show_anim(fig, update, frames=len(alphas), interval=260)

> 🧭 **What you just saw.** The 16 learned numbers form a *coordinate system for atmospheres*:
> nearby codes decode to similar waveforms, and walking from code A to code B morphs the arrival
> pattern continuously — phase first, then amplitude. That is the operational meaning of "the
> encoder learned a representation": the latent space organizes the data according to what
> matters for the task — here, propagation favourability — not according to anything we
> hand-designed.
>
> 📐 **Two honest caveats.** (i) Linear interpolation in code space need not stay on the data
> manifold (H10): the intermediate codes may correspond to no physical atmosphere — the smooth
> decoded waveforms are a property of the decoder's continuity, not evidence that the
> interpolated atmosphere exists. (ii) PCA displays 2 of 16 dimensions, chosen for display, not
> by the network. Interpretability here is *empirical* (correlations, visible structure), never
> guaranteed.

## §4 · The loss, its landscape, and the optimizer race — 19–25 min · ⏱ checkpoint: start §5 at minute 25

The **hyperparameters of the loss** here, in the sense of *choices made about the objective*:

1. **Loss family.** Relative $L^2$ (per atmosphere, all four traces) vs MSE. Relative errors put
   *every atmosphere* on the same footing; MSE chases the loud atmospheres. Since the four traces
   of one atmosphere share a normalization (H11), both are reasonable — but they define different
   goals, and on real (heterogeneous-amplitude) data the choice is an experiment, not a detail.
2. **Weight decay $\lambda$** in the AdamW objective $\mathcal L + \lambda\|\theta\|^2_2$
   (applied to real weights only, never to the complex spectral weights).
3. **Learning rate $\eta$**: chooses the trajectory through the landscape (Lab 1, §4; we reuse the
   two-direction slice and the animated optimizer race, in compact form).

In [ ]:
# ---- 4.1: loss family + weight decay, on this task ----
t0 = time.time()
configs = {'rel-L2, lambda=1e-5': dict(loss_name='rel_l2', wd=1e-5),
           'MSE,    lambda=1e-5': dict(loss_name='l2',     wd=1e-5),
           'rel-L2, lambda=1 (aggressive)': dict(loss_name='rel_l2', wd=1.0)}
sweep2 = {}
for name, kw in configs.items():
    torch.manual_seed(0)
    h, _ = fit_ffno(InfrasoundFFNO(), epochs=60, **kw)
    sweep2[name] = h
    print(f'{name:24s}: final val rel-L2 = {h["val"][-1]:.4f}')
print(f'sweep: {time.time()-t0:.0f} s on {DEVICE}')

fig, ax = plt.subplots(figsize=(7, 3))
for name, h in sweep2.items():
    ax.plot(h['val'], label=name)
ax.set_xlabel('epoch'); ax.set_ylabel('validation rel-L2'); ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_title('Objective changes, same architecture and data')
plt.show()

**Reading the sweep.** (i) The relative $L^2$ reaches a better validation error than MSE on this
task — it refuses to sacrifice the quiet atmospheres. (ii) Weight decay only becomes visible when
pushed to aggressive values ($\lambda=1$ with AdamW — decoupled decay shrinks weights by
$\eta\lambda$ per step, so at this small scale mild $\lambda$ does almost nothing); with this
boundary-layer-perturbed ensemble the aggressive $\lambda$ slightly *hurts*: the fine
profile-to-waveform detail the operator must fit is exactly what a strong shrinkage prior
suppresses. **The honest lesson: never report a hyperparameter sweep without its scale —
$\lambda=10^{-4}$ on one problem and $\lambda=1$ on another can be the same setting, and the
sign of the effect is data-dependent.**

In [ ]:
# ---- 4.2: the loss landscape on a 2-D slice, and the optimizer trajectories ----
def flat(m):
    parts = []
    for q in m.parameters():
        v = q.detach().reshape(-1)
        parts.append(torch.view_as_real(v).flatten() if v.is_complex() else v.flatten())
    return torch.cat([t.cpu() for t in parts])

def load_flat(m, vec):
    i = 0
    for q in m.parameters():
        n = q.numel()
        if q.is_complex():
            q.data.copy_(torch.view_as_complex(vec[i:i + 2 * n].reshape(*q.shape, 2).to(q.device)))
            i += 2 * n
        else:
            q.data.copy_(vec[i:i + n].view_as(q).to(q.device)); i += n

torch.manual_seed(0)
theta_star = flat(ffno)                       # trained operator
theta_init = flat(InfrasoundFFNO())           # same seed -> the exact starting point
d1 = theta_init - theta_star; s0 = d1.norm(); d1 = d1 / s0
g = torch.Generator().manual_seed(1)
d2 = torch.randn_like(d1); d2 = d2 / d2.norm()

xb, yb = Xtr[:16], Ytr[:16]                   # fixed batch: deterministic landscape
probe = InfrasoundFFNO().to(DEVICE)
loss_fn = get_loss('rel_l2')

@torch.no_grad()
def plane_loss(a, b):
    load_flat(probe, theta_star + a * d1 + b * d2)
    return loss_fn(probe(xb), yb).item()

G = 13
alphas = torch.linspace(-1.15 * s0, 1.15 * s0, G)
betas = torch.linspace(-1.15 * s0, 1.15 * s0, G)
Z = np.array([[plane_loss(float(a), float(b)) for b in betas] for a in alphas])
print(f'landscape: {G}x{G} true-loss evaluations; ||theta_init - theta*|| = {s0:.2f}')

def trajectory(kind, lr, momentum=0.0, steps=80):
    load_flat(probe, theta_init)
    opt = {'sgd': lambda: torch.optim.SGD(probe.parameters(), lr=lr, momentum=momentum),
           'adam': lambda: torch.optim.Adam(probe.parameters(), lr=lr)}[kind]()
    path, losses = [], []
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(probe(xb), yb)
        loss.backward(); opt.step()
        v = flat(probe)
        path.append((float((v - theta_star) @ d1), float((v - theta_star) @ d2)))
        losses.append(loss.item())
    return np.array(path), np.array(losses)

runs = {'SGD lr 1e-2': trajectory('sgd', 1e-2),
        'SGD+momentum lr 3e-2': trajectory('sgd', 3e-2, momentum=0.9),
        'Adam lr 2e-3': trajectory('adam', 2e-3)}
for n_, (_, ls_) in runs.items():
    print(f'{n_:24s}: loss {ls_[0]:.4f} -> {ls_[-1]:.5f}')

In [ ]:
# ---- animation 3: the optimizer race on the infrasound landscape ----
fig, ax = plt.subplots(figsize=(6.6, 5))
from matplotlib.colors import LogNorm
im = ax.imshow(Z.T, origin='lower', extent=[float(alphas[0]), float(alphas[-1]),
                                           float(betas[0]), float(betas[-1])],
               cmap='viridis', aspect='auto', norm=LogNorm())
ax.contour(alphas, betas, Z.T, levels=10, colors='white', linewidths=0.5, alpha=0.6)
ax.plot(0, 0, 'r*', ms=15, label=r'trained $\theta^\star$ (section 3)')
trajs = {}
for (name, (path, _)), col in zip(runs.items(), ['C1', 'C2', 'w']):
    (ln,) = ax.plot([], [], color=col, lw=1.6, label=name)
    (pt,) = ax.plot([], [], 'o', color=col, ms=5)
    trajs[name] = (ln, pt)
ax.set_xlabel(r'alpha (direction "from init")'); ax.set_ylabel(r'beta (random direction)')
ax.legend(fontsize=8, loc='upper right'); title = ax.set_title('')

def update(f):
    for name, (ln, pt) in trajs.items():
        path, _ = runs[name]
        ln.set_data(path[:f + 1, 0], path[:f + 1, 1])
        pt.set_data([path[f, 0]], [path[f, 1]])
    title.set_text(f'optimizer step {f + 1}/{len(runs["Adam lr 2e-3"][1])}')

show_anim(fig, update, frames=len(runs['Adam lr 2e-3'][1]), interval=90)

**Reading the race — and it is a striking one.** On *this* landscape, both fixed-step SGD variants
**stall almost completely** (loss $\approx 1.000$ after 80 steps: the network still predicts
"silence"), while **Adam descends to $0.35$ in the same 80 steps**. Why? At initialization this
operator outputs near-zero waveforms, and the relative-$L^2$ surface around that point is
extraordinarily flat for a fixed step size: gradients exist but are tiny and badly scaled, so
$	heta_{t+1} = 	heta_t - \eta
abla\mathcal{L}$ moves at a glacial pace — momentum does not
save it, it only averages an already-tiny direction. Adam rescales each parameter by its own
gradient history (a diagonal preconditioner), which turns the anisotropic, tiny-gradient terrain
into steps of useful size. Compare with Lab 1, where the same SGD happily descended the Burgers
landscape: **"which optimizer works" is a property of the loss geometry, not of the optimizer** —
which is exactly why you should run this 3-minute race on your own problem before trusting any
training curve.

### 4.3 · 🥾 Optional deep dive: the pathologies every ML practitioner meets — demonstrated, not asserted

Machine learning on scientific data has recurring failure modes that are properties of the
*problem*, not bugs of the code. Four of them live in this very task; we demonstrate each one.

(All four demo cells are already executed in the solution notebook; the instructor runs them
live if time allows — the seed-spread cell alone takes seconds on the GPU.)

**(a) Non-convexity.** $\mathcal{L}(\theta)$ is a composition of nonlinear maps: there is no
single minimum to converge to — the landscape of §4.2 showed plateaus and anisotropic valleys.
Practical consequence: **the result depends on the random initialization**. Same data, same
architecture, different seeds — different answers. Honest practice: report a seed spread, never
a single run.

**(b) Data imbalance.** Atmospheres differ in waveform energy (a factor ~2 here; far more on real
data). A plain MSE spends its gradient budget on the loud atmospheres and quietly under-serves
the others, which are equally informative *physically*. Relative losses re-balance; there is no
universally correct choice — only a choice of what "fair" means, stated in the loss.

**(c) Overfitting under small data.** 145k parameters can memorize 8 atmospheres. The
training/validation split exists precisely to catch this: the signature is a training loss that
keeps falling while validation stalls.

**(d) Distribution shift — the geophysical variant.** Every guarantee above is conditional on
H10 (test atmospheres drawn from the training law). Feed the model an atmosphere type never
sampled, and the resulting error measures a *violated hypothesis*, not the method.

In [ ]:
# ---- (a) non-convexity, quantified: five seeds, everything else identical ----
seed_err = []
for sd in range(5):
    torch.manual_seed(sd)
    h_s, _ = fit_ffno(InfrasoundFFNO(), epochs=40, seed=sd)
    seed_err.append(h_s['val'][-1])
    print(f'seed {sd}: final validation rel-L2 = {seed_err[-1]:.4f}')
seed_err = np.array(seed_err)
print(f'spread across seeds: min {seed_err.min():.4f}, max {seed_err.max():.4f} '
      f'({100 * (seed_err.max() - seed_err.min()) / seed_err.mean():.0f}% relative) — '
      'non-convexity, not data noise: the data are fully deterministic')

fig, ax = plt.subplots(figsize=(5.6, 2.9))
ax.bar([f'seed {i}' for i in range(5)], seed_err, color='C0')
ax.axhline(seed_err.mean(), color='C3', ls='--', label='mean over seeds')
ax.set_ylabel('validation rel-L2'); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')
ax.set_title('same data, same architecture: the initialization changes the answer')
plt.show()

In [ ]:
# ---- (b) imbalance: who does each loss serve? quiet vs loud validation atmospheres ----
per_model = {}
for lname, label in [('rel_l2', 'relative-L2 loss'), ('l2', 'MSE loss')]:
    m = InfrasoundFFNO()
    torch.manual_seed(0)
    fit_ffno(m, epochs=40, loss_name=lname)
    with torch.no_grad():
        pred = m(Xva)
    per_model[label] = ((pred.flatten(1) - Yva.flatten(1)).norm(dim=1)
                        / Yva.flatten(1).norm(dim=1)).cpu().numpy()
amp = Yva.flatten(1).norm(dim=1).cpu().numpy()
quiet = amp < np.median(amp)
print(f'validation energy spread: {amp.min():.2f} to {amp.max():.2f} '
      f'(ratio {amp.max() / amp.min():.1f})')
for label, e in per_model.items():
    print(f'{label:18s}: mean relative error on the QUIET half {e[quiet].mean():.4f} | '
          f'on the LOUD half {e[~quiet].mean():.4f}')

fig, ax = plt.subplots(figsize=(6.6, 3.2))
for (label, e), c in zip(per_model.items(), ['C0', 'C3']):
    ax.scatter(amp, e, s=42, color=c, label=label)
ax.axvline(np.median(amp), color='0.6', ls=':', label='median-energy split')
ax.set_xlabel('atmosphere waveform energy (validation)')
ax.set_ylabel('per-atmosphere relative error')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title('imbalance: the MSE-trained model (red) favors the loud atmospheres')
plt.show()

In [ ]:
# ---- (c) overfitting: the same 145k-parameter model trained on EIGHT atmospheres ----
x8, y8 = Xtr[:8], Ytr[:8]
torch.manual_seed(0)
small = InfrasoundFFNO().to(DEVICE)
opt = torch.optim.AdamW(small.parameters(), lr=2e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=120)
loss_fn8 = get_loss('rel_l2')
tr_curve, va_curve = [], []
for ep in range(120):
    small.train()
    opt.zero_grad(set_to_none=True)
    loss = loss_fn8(small(x8), y8)            # the SAME 8 atmospheres at every step
    loss.backward(); opt.step(); sched.step()
    small.eval()
    with torch.no_grad():
        tr_curve.append(loss_fn8(small(x8), y8).item())
        va_curve.append(get_loss('rel_l2')(small(Xva), Yva).item())
print(f'8 training atmospheres: final TRAIN rel-L2 {tr_curve[-1]:.4f} vs VALIDATION '
      f'{va_curve[-1]:.4f} (a gap of x{va_curve[-1] / max(tr_curve[-1], 1e-9):.0f})')

fig, ax = plt.subplots(figsize=(6.4, 3))
ax.plot(tr_curve, label='training loss (8 atmospheres, seen every step)')
ax.plot(va_curve, label='validation rel-L2 (never-seen atmospheres)')
ax.set_yscale('log'); ax.set_xlabel('epoch'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title('memorization: the diverging curves ARE overfitting')
plt.show()

In [ ]:
# ---- (d) distribution shift: an atmosphere from OUTSIDE the ensemble (H10 violated) ----
from fno_tuto.infrasound import parabolic_waveforms
z_out = np.linspace(0.0, CFG.z_max_m, CFG.n_alt)
c_out = ceff[0].numpy().copy()
c_out = c_out + 18.0 * np.exp(-((z_out / 1000.0 - 0.6) / 0.4) ** 2)   # a BL jet far beyond H10
p_out = parabolic_waveforms(c_out, z_out, CFG)
c_norm = torch.tensor((c_out - float(c_mu)) / float(c_sd)).float()[None].to(DEVICE)
with torch.no_grad():
    pred_out = ffno(c_norm)[0].cpu().numpy() * float(p_sd)
rel_out = np.linalg.norm(pred_out - p_out) / np.linalg.norm(p_out)
print(f'out-of-support profile error: {rel_out:.3f} — compare with the in-ensemble test '
      'error of section 5')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(z_km, ceff[0].numpy(), color='C0', label='a training atmosphere')
ax[0].plot(z_km, c_out, color='C3', label='out-of-support profile')
ax[0].set_xlabel('altitude z (km)'); ax[0].set_ylabel('c_eff (m/s)'); ax[0].legend(fontsize=8)
ax[0].set_title('H10 violated: this profile is not from the ensemble')
ax[1].plot(tau_s, p_out[3], color='C0', lw=1.8, label='PE truth (42 km)')
ax[1].plot(tau_s, pred_out[3], color='C3', ls='--', lw=1.6, label='FFNO')
ax[1].set_xlabel('tau (s)'); ax[1].set_ylabel('rel. pressure'); ax[1].legend(fontsize=8)
ax[1].set_title('extrapolation fails, honestly')
plt.show()

**How practitioners answer each pathology.**
- **Non-convexity:** run several seeds, report mean and spread; never compare two
  architectures on a single seed each (five seeds cost seconds on the school GPU).
- **Imbalance:** choose the loss to encode what "fair" means for *your* science — the §4.1
  sweep is the instrument, and per-atmosphere error curves like the one above are the evidence.
- **Overfitting:** watch the train/validation gap; the honest remedies are more data (here the
  PE solver is a free augmentation engine — generate more atmospheres!) and capacity control
  (smaller width, smaller code $k$, weight decay).
- **Distribution shift:** state the data law (H10), test deliberately out-of-support when the
  physics allows it, and report those errors *separately* — never mixed into the test metric.

## §5 · Evaluate on atmospheres the network never saw — 25–30 min · ⏱ wrap-up by minute 30

In [ ]:
# ---- test atmospheres (H10): predicted vs PE truth, all four receivers ----
ffno.eval()
with torch.no_grad():
    pred_te = ffno(Xte).cpu().numpy() * float(p_sd)
truth_te = P[NTR + NVA:].numpy()
rel_te = (np.linalg.norm(pred_te - truth_te, axis=(1, 2))
          / np.linalg.norm(truth_te, axis=(1, 2)))
print(f'test rel-L2 per atmosphere: mean {rel_te.mean():.3f} | worst {rel_te.max():.3f}')

fig, ax = plt.subplots(3, 5, figsize=(14.5, 6.5))
for i in range(3):
    c_phys = Xte[i].cpu().numpy() * float(c_sd) + float(c_mu)
    ax[i, 0].plot(c_phys, z_km, color='C1', lw=1.8)
    ax[i, 0].axhspan(0, 2, color='orange', alpha=0.10)
    ax[i, 0].set_ylim(0, 10); ax[i, 0].set_ylabel('altitude (km)', fontsize=9)
    if i == 2: ax[i, 0].set_xlabel('c_eff (m/s)', fontsize=9)
    ax[i, 0].set_title(f'INPUT: held-out atm {i}', fontsize=9)
    for j in range(4):
        ax[i, 1 + j].plot(tau_s, truth_te[i, j], color='C0', lw=1.8, label='PE truth')
        ax[i, 1 + j].plot(tau_s, pred_te[i, j], color='C3', ls='--', lw=1.4, label='FFNO')
        if i == 0 and j == 0: ax[i, 1 + j].legend(fontsize=8)
        if i == 2: ax[i, 1 + j].set_xlabel('tau (s)', fontsize=9)
        if j == 0: ax[i, 1 + j].set_ylabel('rel. p')
        ax[i, 1 + j].grid(alpha=0.3)
        ax[i, 1 + j].set_title(f'{int(ranges_km[j])} km (rel {rel_te[i]:.2f})', fontsize=9)
fig.suptitle('Held-out atmospheres: the input profile (vertical) and the FOUR ground '
             'receivers predicted jointly from it', y=0.99)
plt.tight_layout(); plt.show()

**What the network did and did not learn** — read the figure honestly:

- **Learned**: travel times (phases), favourable vs unfavourable amplitude patterns, multi-path interference.
- **Bound by hypotheses**: everything is conditional on H1–H13. Change the source, the receiver
  geometry, or leave the synthetic ensemble (H10), and the operator must be retrained — what
  *transfers* is the method, not the weights.

## §6 · Takeaways — and using FFNOs on *your* problem

- **Operator thinking.** You never predicted "a number": input and output were *functions*
  (profile, waveforms), and resolution/grid details stayed out of the parameters (Lab 1, §5).
- **Factorization is a modeling act.** $C^2(m_x+m_y)$ instead of $C^2 m_x m_y$ parameters buys
  depth and speed at the price of separable filters (H13); depth and nonlinearities pay some of it
  back. State it as a hypothesis, test it against the full FNO on a small case.
- **Loss = experiment definition.** Relative vs absolute, weight decay, learning rate: §4 showed
  the geometry they create and why adaptive optimizers fit anisotropic operator landscapes.

**Checklist: Fourier operators on your application.**
1. Write $G^\dagger$ and the *generative* hypotheses of your data honestly (our H1–H8): most
   failures are hypothesis violations, not architecture failures.
2. Choose grids so the informative band fits in few modes; check the spectra yourself (F2).
3. Encoder for the low-dimensional input, FFNO blocks on the output grid — this recipe
   (profile → field) reuses directly for tomography, source inversion, or surrogate solvers.
4. Train with AdamW + cosine; run the 3-minute landscape race (§4) before long trainings.
5. Evaluate on held-out *physical* cases (§5), per-channel and per-case, not only with a scalar.
6. If your geometry is irregular (sensor networks, meshes), go to **Lab 3**: graphs do not need
   a grid at all.

### Optional — save this interactive session measurement (Hedy only)
Identical to Lab 1: writes a JSON timing report on the cluster, a one-line summary elsewhere.

## References

**Operator learning and FNO/FFNO:**
- Z. Li, N. Kovachki, K. Azizzadenesheli, B. Liu, K. Bhattacharya, A. Stuart, A. Anandkumar.
  *Fourier Neural Operator for Parametric Partial Differential Equations.* ICLR 2021.
- Z. Li, N. Kovachki, ... A. Anandkumar. *Neural Operator: Learning Maps Between Function
  Spaces.* Nature Machine Intelligence **6**, 666–676 (2024).
- A. Tran, A. Mathews, L. Xie, C.-K. Ong. *Factorized Fourier Neural Operators.* ICLR 2023.
- K. Azizzadenesheli, Z. Li, ... A. Anandkumar. *Neural Operators for Accelerating Scientific
  Simulations? Open Problems and Perspectives.* ICLR 2024 (position paper).

**Graph neural networks:**
- T. N. Kipf, M. Welling. *Semi-Supervised Classification with Graph Convolutional Networks.*
  ICLR 2017.
- J. Gilmer, S. S. Schoenholz, P. F. Riley, O. Vinyals, G. E. Dahl. *Neural Message Passing for
  Quantum Chemistry.* NeurIPS 2017.
- Q. Li, Z. Han, X.-M. Wu. *Deeper Insights into Graph Convolutional Networks for Semi-
  Supervised Learning.* AAAI 2018. (Oversmoothing and the Dirichlet-energy analysis.)
- R. L. Murphy, B. Srinivasan, V. Rao, B. Ribeiro. *Janossy Pooling: Learning Deep
  Permutation-Invariant Functions.* NeurIPS 2019. (Why permutation-invariant aggregation
  matters.)

**Loss landscapes and optimization:**
- H. Li, Z. Xu, G. Taylor, C. Studer, T. Goldstein. *Visualizing the Loss Landscape of Neural
  Nets.* NeurIPS 2018. (The 2-D random-slice methodology used in Lab 1 §4.)
- T. Garipov, P. Izmailov, D. Podoprikhin, D. Vetrov, A. Wilson. *Loss Surfaces, Mode
  Connectivity, and Fast Ensembling of DNNs.* NeurIPS 2018. (Why two trained networks are
  connected by a low-loss path — the multi-minimum geometry of Lab 1 §4.4.)

**Infrasound propagation and parabolic-equation methods:**
- A. Le Pichon, E. Blanc, A. Hauchecorne (eds.). *Infrasound Monitoring for Atmospheric
  Studies.* 2nd ed., Springer (2021).
- R. Waxler, J. Assink. *Propagation of infrasound through the atmosphere: theory and
  observations.* In: Infrasound Monitoring, ch. 5 (2019).
- D. P. Knupp, S. J. Franke. *A global reference atmosphere for infrasound propagation.*
  J. Acoust. Soc. Am. **140**, 2724 (2016).

**Software:**
- A. Paszke et al. *PyTorch: An Imperative Style, High-Performance Deep Learning Library.*
  NeurIPS 2019.
- M. Fey, J. E. Lenssen. *Fast Graph Representation Learning with PyTorch Geometric.*
  ICLR Workshop on Representation Learning on Graphs and Manifolds (2019).


In [ ]:
if HEDY_SESSION is not None:
    HEDY_SESSION.report()   # writes a JSON timing report on the Hedy cluster only
else:
    print('No Hedy session: nothing to report (laptop run).')